# 使用RAG技术构建企业级文档问答系统之QA抽取

In [2]:
import langchain,langchain_community, pypdf, openai

In [3]:
# 设置全局参数
import os
from dotenv import load_dotenv

load_dotenv()

dirpath = os.getcwd()
outputdir = os.path.join(os.path.join(dirpath, "output"))
os.makedirs(outputdir, exist_ok=True)

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
from uuid import uuid4 # 这个可以生成一串id，用于区分
import pickle

# 提取pdf内容

下面代码就是提取pdf内容，分别封装成uuid2large_doc和uuid2doc
两个分块的大小不一样，每个块包含的内容量是不一样

In [5]:
def split_docs(documents, filepath, chunk_size=400, chunk_overlap=40, seperators = ['\n\n\n', '\n\n'], force_split=False):
    # 如果不是强制切分，可以优先读取缓存
    if os.path.exists(filepath) and not force_split:
        return pickle.load(open(filepath, 'rb')) # 这里在读取filepath的路径的文件
    
    # 创建分词器
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        separators=seperators
    )
    
    # 把文章进行分词
    split_docs = splitter.split_documents(documents)
    
    # 对于每一段词，都生成一个uuid
    for chunk in split_docs:
        chunk.metadata["uuid"] = str(uuid4())
    
    # 写入到指定路径的缓存 
    pickle.dump(split_docs, open(filepath, "wb")) 
    
    return split_docs

In [6]:
# 装载pdf
loader = PyPDFLoader("data/2024全球经济金融展望报告.pdf")
documents = loader.load()

In [21]:
pattern = r"^全球经济金融展望报告\n中国银行研究院 \d+ 2024年"
# 将匹配到的内容替换成None，然后再把结果用\n进行拼接，然后最后存到Document
merged_docs = [Document(page_content='\n'.join(re.sub(pattern, '', doc.page_content) for doc in documents))]
# 调用上面的内容
splitted_docs = split_docs(documents, os.path.join(outputdir, 'split_docs.pkl'), chunk_size=500, chunk_overlap=50)
splitted_docs_large = split_docs(merged_docs, os.path.join(outputdir, 'split_docs_large.pkl'), chunk_size=1500, chunk_overlap=100)

# 然后用uuid作为key，content作为内容，封装doc：

# uuid2doc = {}  # 空字典
# for doc in splitted_docs:  # 遍历所有小粒度文本块
#     key = doc.metadata['uuid']  # 取文本块的唯一ID
#     value = doc.page_content   # 取文本块的实际内容
#     uuid2doc[key] = value      # 把ID和内容存入字典
uuid2doc = {doc.metadata['uuid']: doc.page_content for doc in splitted_docs}
uuid2large_doc = {doc.metadata['uuid']: doc.page_content for doc in splitted_docs_large}

# QA prompt

In [22]:
qa_gen_prompt_tmpl = """
我会给你一段文本（<document></document>之间的部分），你需要阅读这段文本，分别针对这段文本生成8个问题、用户回答这个问题的上下文，和基于上下文对问题的回答。

对问题、上下文、答案的要求：

问题要与这段文本相关，不要询问类似“这个问题的答案在哪一章”这样的问题
上下文：上下文必须与原始文本的内容保持一致，不要进行缩写、扩写、改写、摘要、替换词语等
答案：回答请保持完整且简洁，无须重复问题。答案要能够独立回答问题，而不是引用现有的章节、页码等

返回结果以JSON形式组织，格式为[{"question": "...", "context": ..., "answer": "..."}, ...]。
如果当前文本主要是目录，或者是一些人名、地址、电子邮箱等没有办法生成有意义的问题时，可以返回[]。

下方是文本：
<document>
{{document}}
</document>

请生成结果：
"""

qa_gen_prompt_tmpl_large_context = """
我会给你一段文本（<document></document>之间的部分），你需要阅读这段文本，分别针对这段文本生成2个问题，和基于这段文本对问题的回答，回答请保持完整，无须重复问题。
尽可能创建一些需要综合*大段*文本才能回答的问题，但不要问类似“这一段主要讲了什么内容”这样的问题，答案要能够独立回答问题，而不是引用现有的章节、页码等；不要问具体过于细节的问题，例如“海湾国家的2024年预期经济增长率是多少”，而是尽可能问类似“2024年全球经济的几大趋势是什么”、“受局部中东地区紧张局势影响，可能对全球原物料有哪些影响”。
返回结果以JSON形式组织，格式为[{"question": "...", "answer": "..."}, ...]。
如果当前文本主要是目录，或者是一些人名、地址、电子邮箱等没有办法生成有意义的问题时，可以返回[]。

下方是文本：
<document>
{{document}}
</document>

请生成结果：
"""

In [23]:
from openai import OpenAI
import time 
import random
import threading
import concurrent.futures
from tqdm.auto import tqdm
import json

In [27]:
client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url=os.environ["BASE_URL"]
)


# 这个函数就是把prompt_tmpl替换成text，然后返回
def build_qa_prompt(prompt_tmpl, text):
    prompt = prompt_tmpl.replace('{{document}}', text).strip() 
    return prompt

# 这个函数开始和大模型进行对话。
def chat(prompt,  max_retry=3, debug=False, top_p=0.95, temperature=0.85):

    # 这个函数调用了大模型
    # 进行一次的对话会同时返回多个结果，选取最高概率的第一个。
    def do_chat(prompt):
        completion = client.chat.completions.create(
            model="qwen-long",
            messages=[{"role" : "user", "content":prompt}],
            top_p = top_p, # Top key的值越大，说明会保留的答案路径就越多，回大模型回复的答案就越丰富。
            temperature=temperature
        )
        return completion.choices[0].message.content
    
    # 开始尝试对话，并且尝试多次。如果失败重新尝试就是1~4秒钟。
    while max_retry > 0 : 
        try:
            return do_chat(prompt)
        except Exception as e :
            max_retry = -1
            sleep_seconds = random.randint(1,4)
            if debug : 
                print(f"{str(e)}, remain retry: {max_retry}, sleeping {sleep_seconds}s {prompt}")
            time.sleep(sleep_seconds)
    return None

def gen_qa(splitted_docs, prompt_tmpl, qa_ckpt_filename):
    """
    gen_qa任务就是用多个进程。把documents丢入到llm里面进行处理。返回来的结果是一个字典:{uuid, llm处理后的documents}
    """
    qa_ckpt = {} # 存储llm返回的结果
    
    # 他的任务是读取已经处理好的文结果，然后把它转化成字典
    if os.path.exists(qa_ckpt_filename):
        qa_ckpt = open(qa_ckpt_filename).readlines()
        qa_ckpt = [json.loads(line.strip()) for line in qa_ckpt if line.strip() != '']
        qa_ckpt = {item["uuid"]:item for item in qa_ckpt}
        print(f"found checkpoint, item count : {len(qa_ckpt)}")
    
    # 锁定当前线程。    
    file_lock = threading.Lock()
    max_workers = 4
    
    # 这里创建一个线程池。
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        
        # 这里调用线程处理结果，然后把线程任务存储到futures = {文档UUID: 线程任务}
        futures = {doc.metadata["uuid"]:executor.submit(chat, build_qa_prompt(prompt_tmpl, doc.page_content), 3, True) for doc in splitted_docs if len(doc.page_content.replace("\n", "")) >= 150 and doc.metadata["uuid"] not in qa_ckpt}
        
        for uuid in tqdm(futures) : 
            # 拿到UUID对应的[线程任务]
            future = futures[uuid]
            
            # 等待线程完成阻塞在这里, future线程执行结果返回
            result = future.result()
            
            if result is None:
                continue
            
            item = {"uuid" : uuid, "raw_resp" : result}
            
            # 这里获取锁，防止多进程写入同一个文件。
            file_lock.acquire()

            try :
                # a 在文件的末尾添加内容，不删除、不覆盖原来的内容
                with open(qa_ckpt_filename, "a") as f:
                    f.write(json.dumps(item, ensure_ascii=False)+'\n')
            except Exception as e :
                print(e)
            # 不管报错与否，最后都会执行释放锁。
            finally:
                file_lock.release()
    return qa_ckpt 

In [28]:
# 短上下文抽取结果
detailed_qa_dict = gen_qa(splitted_docs, qa_gen_prompt_tmpl, os.path.join(outputdir, f"qa_ckpt_detailed.jsonl"))
# # 长上下文抽取结果，这里llm调用不接受所以没有使用
# large_context_qa_dict = gen_qa(splitted_docs_large, qa_gen_prompt_tmpl_large_context, os.path.join(outputdir, f"qa_ckpt_large_context.jsonl"))

found checkpoint, item count : 50


0it [00:00, ?it/s]


In [39]:
# 现在打印出来的就只有 2 条数据了，不会刷屏！
print(json.dumps(dict(list(detailed_qa_dict.items())[:3]), indent=4, ensure_ascii=False))

{
    "9f9d826d-70b0-4e14-b15c-ab0a966e734d": {
        "uuid": "9f9d826d-70b0-4e14-b15c-ab0a966e734d",
        "raw_resp": "[\n  {\n    \"question\": \"2023年全球经济增长动力呈现怎样的趋势？\",\n    \"context\": \"● 2023 年全球经济增长动力持续回落，各国复苏分化，发达经济体增速明显放缓，新兴经济体整体表现稳定。\",\n    \"answer\": \"2023年全球经济增长动力持续回落。\"\n  },\n  {\n    \"question\": \"2023年发达经济体和新兴经济体的经济增速表现有何差异？\",\n    \"context\": \"● 2023 年全球经济增长动力持续回落，各国复苏分化，发达经济体增速明显放缓，新兴经济体整体表现稳定。\",\n    \"answer\": \"发达经济体增速明显放缓，新兴经济体整体表现稳定。\"\n  },\n  {\n    \"question\": \"2023年全球贸易增长状况如何？\",\n    \"context\": \"● 2023 年全球经济增长动力持续回落，各国复苏分化，发达经济体增速明显放缓，新兴经济体整体表现稳定。全球贸易增长乏力，各国生产景气度逐渐回落，内需对经济的拉动作用减弱。\",\n    \"answer\": \"全球贸易增长乏力。\"\n  },\n  {\n    \"question\": \"2023年欧美央行货币政策紧缩态势有何变化？\",\n    \"context\": \"● 2023 年全球经济增长动力持续回落，各国复苏分化，发达经济体增速明显放缓，新兴经济体整体表现稳定。全球贸易增长乏力，各国生产景气度逐渐回落，内需对经济的拉动作用减弱。欧美央行货币政策紧缩态势放缓，美元指数高位震荡后走弱，全球股市表现总体好于预期，但区域分化明显。\",\n    \"answer\": \"欧美央行货币政策紧缩态势放缓。\"\n  },\n  {\n    \"question\": \"2023年高利率环境对债券市场产生了什么影响？\",\n    \"context\

# 后处理

这里后处理就是把llm的数据进行清洗， 然后把每一条的信息存储到字典当中， 最后返回字典

In [41]:
import re
import pandas as pd

def convert2json(text):
    # 定义规则：匹配 中括号[] 包裹的所有内容（AI生成的问答都在[]里）
    pattern = r'\[.*\]'

    # 把文本里多余的 >>> 符号删掉（AI乱输出的垃圾字符）
    text = text.replace('>>>', '')
    try:
        # 尝试直接把文本转成JSON格式（标准数据格式）
        return json.loads(text)
    except:
        # 直接转换失败 → 用正则提取文本里的 [] 内容
        match = re.search(pattern, text, re.DOTALL)
        try:
            # 拿到提取到的[]字符串
            matched = match.group(0)
            # 把提取的内容转成JSON
            return json.loads(matched)
        except Exception as e:
            # 彻底失败，打印错误
            print(f"{match}, {str(e)}")

    # 所有方法都失败 → 返回空列表
    return []


def build_qa_df(qa_ckpt, uuid2doc_map):
    data = []
    
    for key, value in tqdm(qa_ckpt.items()):
        
        # 取出llm取出来的字段
        text = value["raw_resp"]
        uuid = key 
        
        # 清洗llm输出的内容。把一些不好用的东西给清洗掉
        qa_list = convert2json(text)
        
        for item in qa_list:
            question = item.get("question", "").strip()
            answer = item.get("answer", "").strip()
            context = item.get("context", "").strip()
            
            if question == "" or answer == "" :
                print(qa_list)
                continue
            data.append({
                'uuid': key,
                'question': question,
                'answer': answer,
                'context': context,
                'doc': uuid2doc_map[key]
            })
    qa_df = pd.DataFrame(data)
    return qa_df

In [42]:
# 把llm返回的数据封装成字典
qa_df = build_qa_df(detailed_qa_dict, uuid2doc) 

qa_df.drop_duplicates("question", inplace=True)

# 这里是Dataframe格式，他可以自动给每一行的qa_type变成 detailed
qa_df["qa_type"] = "detailed"

100%|██████████| 50/50 [00:00<00:00, 6625.24it/s]


# QA质量打分prompt

In [47]:
qa_check_prompt_tmpl = """
你是一个金融领域的专家，现在有人根据一份经济发展报告，构造了一些问题，并对问题进行了回答。
你的任务是对这些问题（<question></question>之间的部分）和回答（<answer></answer>）进行打分。

结果请以JSON形式组织，格式如下（<result></result>之间的部分）：
<result>
{"score": ..., "reason": ...}
</result>
其中score是对问题-回答的打分，分值是一个int类型的值，取值范围为1-5。reason是打分的理由。

好的问题，应该是询问事实、观点等，不好的问题，通常要求做一些文本摘要等初级文字处理工作，类似于“这一段描述了什么”，“文本描述了什么”；或者询问的内容是图相关的，例如“图4展示了什么数据？”。
好的答案，应该能够回应问题，而不是回答无关的内容，不好的回答，会给出在原文中的引用，例如“第3章”等。

问题：
<question>
{{question}}
</question>

参考答案：
<answer>
{{answer}}
</answer>

请进返回JSON格式的数据即可，不要添加其他任何描述性信息。
"""

In [48]:
# 设置下llm的对应prompt
def build_qa_scoring_prompt(row):
    context = row["context"]
    question = row["question"]
    answer = row["answer"]
    prompt = qa_check_prompt_tmpl.replace("{{question}}", question).replace("{{answer}}", answer)
    
    return prompt


在大规模打分之前，先调试一下，检查Prompt是否正常

In [49]:

qa_df.iloc[41].to_dict()

{'uuid': '82264b1c-9800-4e31-bb99-507bd10ac6f2',
 'question': '2023年二季度欧元区固定资本形成总额环比增长率是多少？',
 'answer': '0.1%',
 'context': '2023 年二季度，欧元区固定资本形成总额环比增长 0.1%，比一季度增速下降 0.3 个百分点，房地产对 GDP 环比增长拉动率转为负值。',
 'doc': '全球经济金融展望报告\n中国银行研究院 5 2024 年\n图 4：部分欧洲国家零售销售指数\n注：除英国是以 2019 年为基年外，其他经济体均为 2015 年为基年。\n资料来源：Wind，中国银行研究院\n发达经济体投资受加息政策影响较大，国内投资和跨境投资均持续承压。\n美国私人投资在 2023 年一季度触底后逐渐反弹，三季度存货及住宅投资恢复增\n长，带动私人投资增速提升至 8.4%（经季调后环比折年率），但制造业和设备\n投资均放缓，环比增长折年率分别降低 0.1%和 3.8%。欧盟投资增速放缓，房\n地产投资减少。2023 年二季度，欧元区固定资本形成总额环比增长 0.1%，比一\n季度增速下降 0.3 个百分点，房地产对 GDP 环比增长拉动率转为负值。在紧缩\n货币政策影响下，发达经济体企业部门宏观杠杆率下降，企业加杠杆或负债投\n资意愿不足。同 2022 年底相比，2023 年二季度，美国、英国、法国、意大利\n和德国非金融企业部门负债率分别下降了 2.4 个、3.4 个、4.0 个、3.0 个和 1.3\n个百分点（图 5）。IMF 预测 2023 年全球投资率（投资占 GDP 的比重）将下\n降 1.0 个百分点至 26.4%（图 6），其中，欧盟将下降 1.1 个百分点，比发达经\n济体平均降幅高 0.2 个百分点。从跨境投资角度看，受地缘政治局势紧张、金\n融领域动荡加剧、高利率和投资审查趋严等影响，并购交易仍然疲软，而在全\n球产业链重塑背景下，东南亚等区域绿地投资恢复增长。联合国贸发会议预计',
 'qa_type': 'detailed'}

In [50]:

print(build_qa_scoring_prompt(qa_df.iloc[41]))


你是一个金融领域的专家，现在有人根据一份经济发展报告，构造了一些问题，并对问题进行了回答。
你的任务是对这些问题（<question></question>之间的部分）和回答（<answer></answer>）进行打分。

结果请以JSON形式组织，格式如下（<result></result>之间的部分）：
<result>
{"score": ..., "reason": ...}
</result>
其中score是对问题-回答的打分，分值是一个int类型的值，取值范围为1-5。reason是打分的理由。

好的问题，应该是询问事实、观点等，不好的问题，通常要求做一些文本摘要等初级文字处理工作，类似于“这一段描述了什么”，“文本描述了什么”；或者询问的内容是图相关的，例如“图4展示了什么数据？”。
好的答案，应该能够回应问题，而不是回答无关的内容，不好的回答，会给出在原文中的引用，例如“第3章”等。

问题：
<question>
2023年二季度欧元区固定资本形成总额环比增长率是多少？
</question>

参考答案：
<answer>
0.1%
</answer>

请进返回JSON格式的数据即可，不要添加其他任何描述性信息。



In [60]:
print(chat(build_qa_scoring_prompt(qa_df.iloc[41]), debug=True, temperature=0))

<result>
{"score": 5, "reason": "问题明确询问2023年二季度欧元区固定资本形成总额的环比增长率，属于具体、可验证的经济事实性问题；答案简洁准确，直接给出数值'0.1%'，无冗余信息或无关引用，完全回应问题，符合金融领域数据查询的规范要求。"}
</result>


In [58]:
import threading
import concurrent.futures
from tqdm.auto import tqdm 
import json

# qa scoring ckpt 
qa_scoring_ckpt = {}
qa_scoring_ckpt_filename = os.path.join(outputdir, f"qa_scoring_ckpt.jsonl")
if os.path.exists(qa_scoring_ckpt_filename):
    qa_scoring_ckpt = open(qa_scoring_ckpt_filename).readlines()
    qa_scoring_ckpt = [json.loads(line.strip()) for line in qa_scoring_ckpt if line.strip() != '']
    qa_scoring_ckpt = {item['question']: item for item in qa_scoring_ckpt}
    print(f'found checkpoint, item count: {len(qa_scoring_ckpt)}')
    
file_lock = threading.Lock()
max_workers = 3
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {row["question"]: executor.submit(chat, build_qa_scoring_prompt(row), 3, True, 0) for _,row in qa_df.iterrows() if row["question"] not in qa_scoring_ckpt}
    for question in tqdm(futures):
        # 拿到需要执行的任务
        future = futures[question]
        # 直接执行，拿到结果
        result = future.result()
        if result is None :
            continue
        
        item = {"question" : question, "raw_resp":result}
        qa_scoring_ckpt[question] = item
        
        global file_lock
        file_lock.acquire()
        
        try:
            with open(qa_scoring_ckpt_filename, 'a') as f:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        except Exception as e:
            print(e)
        finally:
            file_lock.release()     

found checkpoint, item count: 392


0it [00:00, ?it/s]


In [ ]:
def extract_json(text):
    """
    从raw_resp中提取出打分结果的JSON，然后返回成字典
    :param raw_resp: LLM的返回结果，数据样例：
    {\n"score": 5, \n"reason": "问题提出了一个具体且明确的观点询问，即针对美国房地产市场的风险评估，特别是对中小型银行的影响。参考答案直接回答了问题，并提供了详细的解释和原因，没有引用原始文本的位置，而是直接给出了分析性的内容。"\n}
    Based on the given criteria:\n\n- The question asks for a factual assessment of global trade performance in 2023, which is clear and direct.\n- The provided answer directly addresses the question without referring back to a source or chapter, offering an evaluation of global trade conditions in 2023.\n\nBoth the question and the answer meet the criteria for being high-quality and appropriately structured. Therefore, I would give them a high score.\n\n```json\n{"score": 5, "reason": "The question is clear and direct, seeking a factual evaluation. The answer provides a direct response without referring to specific sources or sections."}\n```
    """
    # pattern = r'\n```json\n(.*?)\n```'
    # 因为你的打分结果就是 {"score":5, "reason":"xxx"} 这种格式
    pattern = r'\{.*?\}'
    
    ret = {}
    try:
        # 提取到字符串格式的JSON → 转换成 Python字典
        ret = json.loads(text)
    except:
        # 匹配 大括号 {} 包裹的所有内容（就是你要的打分 JSON：{"score":5,"reason":"xxx"}）
        match = re.search(pattern, text, re.DOTALL)
        try:
            # 拿到匹配到的内容（就是 {"score":5, "reason":"xxx"} 这段字符串）
            matched = match.group(0)
            # 提取到字符串格式的JSON → 转换成 Python字典
            ret = json.loads(matched)
        except Exception as e:
            print(f"{match}, {str(e)}")       
    return ret

In [ ]:
qa_scoring_dict = {}

for key, value in qa_scoring_ckpt.items():
    try:
        # 这里key是question，value["raw_resp"]是{score，reason} 然后经过extract_json直接存储
        qa_scoring_dict[key] = extract_json(value['raw_resp'])
        if 'score' not in qa_scoring_dict[key]:
            qa_scoring_dict[key] = {'score': -1, 'reason': f"parse failed, raw resp: {value['raw_resp']}"}
            raise ValueError(f'no score in result, question: {key}')
    except Exception as e:
        print(f"{key}, error: {e}")

In [63]:
# 对于每一个qa_df["question"]，都从qa_scoring_dict当中拿到对应的分数，存储到qa_df["score"]中
qa_df["score"] = qa_df["question"].apply(lambda q : qa_scoring_dict.get(q, {}).get("score", -1))
qa_df["reason"] = qa_df["question"].apply(lambda q: qa_scoring_dict.get(q, {}).get("reason", -1))

In [64]:

qa_df.sample(5)

,uuid,question,answer,context,doc,qa_type,score,reason
179,424144dd-be0f-42c1-a887-877639b437c2,2024年欧元区的CPI涨幅预测值是多少？,2.7%,表 1：2024 年全球主要经济体关键指标预测（%）\n地区 年\n国家\nGDP 增长率 ...,全球经济金融展望报告\n中国银行研究院 22 2024 年\n表 1：2024 年全球主要经...,detailed,5,问题明确询问2024年欧元区CPI涨幅的预测值，属于具体事实性问题；答案简洁准确，直接给出数...
151,14afb073-f335-47c4-949b-aec2f3a298bd,支撑亚太新兴经济体增长的主要因素是什么？,强劲的内需,不过强劲的内需是支撑亚太,全球经济金融展望报告\n中国银行研究院 18 2024 年\n图 12：日本通胀同比增速变化...,detailed,5,问题聚焦于经济发展报告中的具体事实性内容（支撑亚太新兴经济体增长的主要因素），属于高质量的事...
25,c4881eda-0131-4fd7-9877-f6788c5e97d1,全球工业生产量在2023年4月达到什么状态？,触及年内低位。,荷兰经济分析局数据显示，全球工业生产量于 4 月触及年内低位，5-8 月逐月回升，但发达经济...,全球经济金融展望报告\n中国银行研究院 3 2024 年\n图 2：主要经济体 GDP 增速...,detailed,4,问题聚焦于具体时间点（2023年4月）和具体经济指标（全球工业生产量）的状态，属于事实性询问...
361,f1bbf092-cf76-4016-ae43-762ef720eb38,2023年二季度美国家庭全部债务支出占可支配收入的比例是多少？,9.8%。,第二，美国家庭全部债务支出/可支配收入在疫情发生后从 8.3%上升至 2022 年四季度的 ...,全球经济金融展望报告\n中国银行研究院 46 2024 年\n高级信贷官员的考察发现，银行对...,detailed,5,问题明确询问2023年二季度美国家庭全部债务支出占可支配收入的具体数值比例，属于事实性、可验...
214,5ff28a1c-5ff3-45bb-8973-bb8f2fad3022,2023年前9个月，美国货币市场基金净值从多少增至多少？,由 52879 亿美元增至 61531 亿美元,2023 年前 9 个月，美国货币市场基金净值由 52879 亿美元增至 61531 亿美元。,全球经济金融展望报告\n中国银行研究院 26 2024 年\n动性上升，公司债券和公司信贷市...,detailed,5,问题明确询问2023年前9个月美国货币市场基金净值的具体起止数值，属于事实性问题；回答精准给...


In [65]:
# 统计下每个分数有多少个
qa_df['score'].value_counts()

score
5    331
4     49
3      8
2      3
1      1
Name: count, dtype: int64

In [67]:
qa_df[qa_df['score'] == 2]

,uuid,question,answer,context,doc,qa_type,score,reason
53,8f1d42e8-ebb3-4ca4-9e87-a67b1bdeef91,图5的数据来源是哪两个机构？,IIF和中国银行研究院。,图 5：部分发达国家非金融企业部门债务率（%）\n资料来源：IIF，中国银行研究院,全球经济金融展望报告\n中国银行研究院 6 2024 年\n2023 年全球跨境直接投资将继...,detailed,2,问题涉及图表（图5）的数据来源，属于图相关的问题，不符合‘好的问题’标准；虽然答案简洁准确，...
54,8f1d42e8-ebb3-4ca4-9e87-a67b1bdeef91,图6的数据来源是哪两个机构？,IMF和中国银行研究院。,图 6：全球投资率变化趋势（%）\n资料来源：IMF，中国银行研究院,全球经济金融展望报告\n中国银行研究院 6 2024 年\n2023 年全球跨境直接投资将继...,detailed,2,问题询问图6的数据来源，属于图相关的提问，不符合好问题的标准；答案虽然简洁，但问题本身因涉及...
279,db0e86b9-1146-4736-b517-11b93ad05335,哪些国家或地区因拥有大量美元债务且受地缘政治环境变化影响，发生主权债务危机的概率大大提升？,部分发展中国家,此外，部分发展中国家拥有大量美元债务，地缘政治环境变化使其对外偿付能力受损，发生主权债务危机...,全球经济金融展望报告\n中国银行研究院 35 2024 年\n体国债收益率出现较大幅度下跌。...,detailed,2,问题本身是合理的，聚焦于特定经济风险因素（美元债务+地缘政治）与主权债务危机的关联，属于事实...


In [74]:
hq_qa_df = qa_df[qa_df['score'] >= 4]
# 把问题提取出来， 转化成列表
test_q = hq_qa_df.sample(100, replace=False)['question'].values.tolist()

# 把全部添加一个训练样本的字段
hq_qa_df['dataset'] = 'train'

# 把test_q 这几个问题的sample的dataset字段修改成test
hq_qa_df.loc[hq_qa_df['question'].isin(test_q), 'dataset'] = "test"

# 把结果保存到excel中
hq_qa_df.to_excel(os.path.join(outputdir, 'question_answer.xlsx'), index=False)


In [75]:
hq_qa_df.sample(10)

,uuid,question,answer,context,doc,qa_type,score,reason,dataset
24,c4881eda-0131-4fd7-9877-f6788c5e97d1,截至2023年10月底，纽约联储全球供应链压力指数处于什么水平？,降至有记录以来的最低值。,从生产端看，全球供应链持续恢复，但生产景气度逐渐回落。截至 2023 年 10 月底，纽约联...,全球经济金融展望报告\n中国银行研究院 3 2024 年\n图 2：主要经济体 GDP 增速...,detailed,5,问题明确询问具体时间点（2023年10月底）下纽约联储全球供应链压力指数的具体水平，属于事实...,test
58,8cfa0157-1660-4445-b02f-69b5718d943b,2023年10月韩国出口同比增长多少？这是自何时以来的首次正增长？,5.1%，自2022年10月以来,10 月，韩国出口同比增长 5.1%，是自 2022 年 10 月以来首次正增长。,全球经济金融展望报告\n中国银行研究院 7 2024 年\n（图 7）。10 月，世贸组织将...,detailed,5,问题明确询问具体数值和时间点，属于事实性问题，符合金融领域专业提问标准；答案精准回应了两个子...,test
246,2c02441c-8dd4-4e0d-8941-e35fb0e6aea2,日本央行2024年在货币政策方面的探索方向是什么？,审慎探索逐步退出超宽松货币政策。,随着日本央行审慎探索逐步退出超宽松货币政策，预计 2024 年日元或将呈现升值态势。,全球经济金融展望报告\n中国银行研究院 30 2024 年\n图 18：主要货币兑美元汇率较...,detailed,5,问题聚焦于具体国家（日本）、具体机构（日本央行）、具体年份（2024年）和具体政策领域（货币...,train
65,1e665ef7-fc04-416a-924b-77339a99ffb9,2024年美国经济面临的主要风险是什么？,“软着陆”和衰退风险同时存在。,美国经济处于下行阶段，2024 年经济“软着陆”和衰退风险同时存在，但目前来看，经济实现“软...,全球经济金融展望报告\n中国银行研究院 8 2024 年\n率政策。\n二是各国经济走势将进...,detailed,4,问题聚焦于2024年美国经济面临的具体风险，属于事实性、政策/经济分析类的高质量问题；答案简...,test
358,81d7d939-5e7f-4cf4-8916-994d0272b6d4,2023年二季度美国居住房地产市值余额是多少？,56.3 万亿美元。,2023 年二季度，美国居住房地产市值余额为 56.3 万亿美元，不仅远大于商业房地产的 2...,全球经济金融展望报告\n中国银行研究院 45 2024 年\n第四，中海深化经贸合作将为人民...,detailed,5,问题明确询问2023年二季度美国居住房地产市值余额这一具体数值型事实，属于高质量的事实性问题...,test
25,c4881eda-0131-4fd7-9877-f6788c5e97d1,全球工业生产量在2023年4月达到什么状态？,触及年内低位。,荷兰经济分析局数据显示，全球工业生产量于 4 月触及年内低位，5-8 月逐月回升，但发达经济...,全球经济金融展望报告\n中国银行研究院 3 2024 年\n图 2：主要经济体 GDP 增速...,detailed,4,问题聚焦于具体时间点（2023年4月）和具体经济指标（全球工业生产量）的状态，属于事实性询问...,train
371,cb37486f-5603-40d1-9f5d-de376e2e51b7,过去两年美国居住房地产价格经历了怎样的变化？,经历了两位数增长。,过去两年，美国居住房地产价格经历了两位数增长。,全球经济金融展望报告\n中国银行研究院 47 2024 年\n第七，美联储加息以来住房贷款质...,detailed,5,问题聚焦于具体事实（美国居住房地产价格过去两年的变化），具有明确的时间范围和对象；答案简洁准...,train
287,b20a4219-156d-4158-b94e-6b1d3c56e156,布伦特油价在2023年9月27日达到的年内最高点是多少？,96.6美元/桶,WTI 油价和布伦特油价分别于 9 月 27 日升至年内最高点 93.7 美元/桶和 96....,全球经济金融展望报告\n中国银行研究院 36 2024 年\n（六）大宗商品市场：全球大宗商...,detailed,5,问题明确询问具体事实（布伦特油价在特定日期的年内最高点），具有唯一可验证的答案；回答精准给出...,train
185,ba26a011-9e62-415b-86d1-2e84c6b6f89e,2023年美联储将联邦基金利率目标区间上调至多少？,5.25%-5.5%,2023 年，为继续遏制高通胀，主要发达经济体货币政策延续收紧态势，但总体步伐逐步放缓，目前...,全球经济金融展望报告\n中国银行研究院 23 2024 年\n图 14：影响国际金融动态的六...,detailed,5,问题明确询问2023年美联储联邦基金利率目标区间的具体数值，属于典型的事实性问题，聚焦于可验...,train
130,d28f9774-c856-4966-b52b-e76181cd0253,被认定为不合作企业的中国出口商，在欧盟反补贴调查中将面临何种税率？,被征收最高税率。,在此情况下，欧盟一般可能采取两类反补贴措施：一是加征反补贴税，针对不同类型的企业，适用的加征...,全球经济金融展望报告\n中国银行研究院 16 2024 年\n存在专向性\n1\n或属于禁止...,detailed,5,问题聚焦于具体政策后果（不合作企业面临的税率），属于可验证的事实性问题；答案简洁准确，直接回...,train
